In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [2]:
df=pd.read_csv(r'C:\Users\johnf\forks\Statistical-Learning-Lab-Test\ai4i2020.csv')
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [ ]:
df=df.drop(columns=['UDI','Product ID','TWF','HDF','PWF','OSF','RNF'])

target_col = "Machine failure"
cat_col = ["Type"]
num_cols = [col for col in df.columns if col not in cat_col + [target_col]]
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df[target_col])

train_df = train_df[train_df[target_col] == 0]

In [4]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(sparse_output=False), cat_col)
])

X_train = preprocessor.fit_transform(train_df.drop(columns=[target_col]))
X_test = preprocessor.transform(test_df.drop(columns=[target_col]))

y_test = test_df[target_col].values

In [5]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_tensor), batch_size=64, shuffle=True)

In [6]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )
    
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Autoencoder(input_dim=X_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for (batch,) in train_loader:
        batch = batch.to(device)
        
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 40.4704
Epoch 2, Loss: 7.9823
Epoch 3, Loss: 4.4763
Epoch 4, Loss: 2.2542
Epoch 5, Loss: 0.9683
Epoch 6, Loss: 0.2905
Epoch 7, Loss: 0.1645
Epoch 8, Loss: 0.1307
Epoch 9, Loss: 0.1116
Epoch 10, Loss: 0.0930
Epoch 11, Loss: 0.0822
Epoch 12, Loss: 0.0715
Epoch 13, Loss: 0.0603
Epoch 14, Loss: 0.0539
Epoch 15, Loss: 0.0531
Epoch 16, Loss: 0.0449
Epoch 17, Loss: 0.0435
Epoch 18, Loss: 0.0398
Epoch 19, Loss: 0.0370
Epoch 20, Loss: 0.0311
Epoch 21, Loss: 0.0313
Epoch 22, Loss: 0.0308
Epoch 23, Loss: 0.0284
Epoch 24, Loss: 0.0253
Epoch 25, Loss: 0.0264
Epoch 26, Loss: 0.0263
Epoch 27, Loss: 0.0226
Epoch 28, Loss: 0.0206
Epoch 29, Loss: 0.0191
Epoch 30, Loss: 0.0188
Epoch 31, Loss: 0.0239
Epoch 32, Loss: 0.0160
Epoch 33, Loss: 0.0163
Epoch 34, Loss: 0.0185
Epoch 35, Loss: 0.0121
Epoch 36, Loss: 0.0119
Epoch 37, Loss: 0.0164
Epoch 38, Loss: 0.0150
Epoch 39, Loss: 0.0124
Epoch 40, Loss: 0.0130
Epoch 41, Loss: 0.0115
Epoch 42, Loss: 0.0136
Epoch 43, Loss: 0.0093
Epoch 44, Loss: 0.0

In [8]:
model.eval()

with torch.no_grad():
    X_test_device = X_test_tensor.to(device)
    recon = model(X_test_device)
    
    # MSE per sample
    errors = torch.mean((recon - X_test_device) ** 2, dim=1).cpu().numpy()

threshold = np.percentile(errors, 95)  # tune this

with torch.no_grad():
    train_recon = model(X_train_tensor.to(device))
    train_errors = torch.mean((train_recon - X_train_tensor.to(device))**2, dim=1).cpu().numpy()

threshold = np.percentile(train_errors, 95)

In [9]:
y_pred = (errors > threshold).astype(int)

In [10]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, errors))

              precision    recall  f1-score   support

           0       0.97      0.95      0.96      1932
           1       0.17      0.31      0.22        68

    accuracy                           0.92      2000
   macro avg       0.57      0.63      0.59      2000
weighted avg       0.95      0.92      0.94      2000

ROC-AUC: 0.7565308732188528
